# Rebuilt three CatBoost-based submissions

This notebook reconstructs the exact branch that produced:

1. `submission_catboost_annual_calibrated.csv`
2. `submission_catboost_tipologia.csv`
3. `submission_ensemble_B75_C25.csv`

**Important:** these three variants use the 2024–2025 annual Tipologia information and/or annual totals. They are included to reproduce the leaderboard submissions, not as the leakage-free direct-forecasting approach.

Run every cell from top to bottom in a fresh Kaggle session. The last cell creates base64 download buttons, avoiding the `FileLink` 404 problem.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

In [ ]:
DATA_DIR = Path("/kaggle/input")

for path in DATA_DIR.rglob("*"):
    if path.is_file():
        print(path)

In [ ]:
train_path = Path("/kaggle/input/datasets/denisebrancaleone/y-sis-hackathon/train.csv")
test_path = Path("/kaggle/input/datasets/denisebrancaleone/y-sis-hackathon/test.csv")
metadata_path = Path("/kaggle/input/datasets/matascerda/new-metadata/road_metadata (1).csv")
accidents_path = Path("/kaggle/input/datasets/denisebrancaleone/y-sis-hackathon/incidenti.xlsx")

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
metadata = pd.read_csv(metadata_path)

print("Train:", train.shape)
print("Test:", test.shape)
print("Metadata:", metadata.shape)

display(train.head(10))
train.info()

In [ ]:
assignment_columns = [
    "strada_sintetica",
    "Strada",
    "Provincia",
    "Localita",
    "Gruppo",
    "Data Ins"
]

available_columns = [
    column for column in assignment_columns
    if column in train.columns
]

assignment_audit = pd.DataFrame({
    "non_missing": train[available_columns].notna().sum(),
    "missing": train[available_columns].isna().sum(),
    "missing_percentage": (
        train[available_columns].isna().mean() * 100
    ).round(2),
    "unique_values": train[available_columns].nunique(dropna=True)
})

display(assignment_audit)

In [ ]:
has_synthetic_road = train["strada_sintetica"].notna()
has_original_road = train["Strada"].notna()
has_province = train["Provincia"].notna()
has_locality = train["Localita"].notna()

assignment_groups = pd.Series(
    np.select(
        [
            has_synthetic_road,
            ~has_synthetic_road & has_original_road,
            ~has_synthetic_road & ~has_original_road & has_province,
            ~has_synthetic_road & ~has_original_road & ~has_province & has_locality
        ],
        [
            "A_direct_synthetic_road",
            "B_original_road_only",
            "C_province_only",
            "D_locality_only"
        ],
        default="E_no_geographic_information"
    )
)

display(
    assignment_groups
    .value_counts(dropna=False)
    .rename_axis("assignment_group")
    .reset_index(name="number_of_cases")
)

In [ ]:
road_group = (
    train.loc[
        train["strada_sintetica"].notna()
        & train["Gruppo"].notna(),
        ["strada_sintetica", "Gruppo"]
    ]
    .drop_duplicates()
)

groups_per_road = (
    road_group
    .groupby("strada_sintetica")["Gruppo"]
    .nunique()
    .sort_values(ascending=False)
)

print("Roads analysed:", len(groups_per_road))

print(
    "Roads associated with exactly one Gruppo:",
    (groups_per_road == 1).sum()
)

print(
    "Roads associated with more than one Gruppo:",
    (groups_per_road > 1).sum()
)

display(groups_per_road.value_counts().sort_index())

In [ ]:
roads_multiple_groups = groups_per_road[
    groups_per_road > 1
].index

display(
    road_group[
        road_group["strada_sintetica"].isin(roads_multiple_groups)
    ]
    .sort_values(["strada_sintetica", "Gruppo"])
    .head(100)
)

In [ ]:
print(test.shape)
display(test.head(10))

In [ ]:
def normalize_road(series):
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .str.replace('""', '"', regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )

test_roads = set(normalize_road(test["strada"]).dropna())
original_roads = set(normalize_road(train["Strada"]).dropna())
synthetic_roads = set(normalize_road(train["strada_sintetica"]).dropna())

print("Unique roads in test:", len(test_roads))
print()

print("Historical Strada:")
print("Unique values:", len(original_roads))
print("Matches with test:", len(test_roads & original_roads))
print(
    "Percentage of test roads matched:",
    round(100 * len(test_roads & original_roads) / len(test_roads), 2),
    "%"
)
print()

print("Historical strada_sintetica:")
print("Unique values:", len(synthetic_roads))
print("Matches with test:", len(test_roads & synthetic_roads))
print(
    "Percentage of test roads matched:",
    round(100 * len(test_roads & synthetic_roads) / len(test_roads), 2),
    "%"
)

In [ ]:
unmatched_test = sorted(test_roads - original_roads)
historical_not_in_test = sorted(original_roads - test_roads)

print("Test roads without direct historical match:", len(unmatched_test))
display(
    pd.DataFrame({"unmatched_test_road": unmatched_test}).head(50)
)

print("Historical roads not appearing directly in test:", len(historical_not_in_test))
display(
    pd.DataFrame({
        "historical_road_not_in_test": historical_not_in_test
    }).head(50)
)

In [ ]:
# Keep only records where both road variables are available
road_family_pairs = (
    train.loc[
        train["Strada"].notna()
        & train["strada_sintetica"].notna(),
        ["Strada", "strada_sintetica"]
    ]
    .drop_duplicates()
)

# Number of synthetic families associated with each detailed road
families_per_road = (
    road_family_pairs
    .groupby("Strada")["strada_sintetica"]
    .nunique()
)

# Number of detailed roads contained in each synthetic family
roads_per_family = (
    road_family_pairs
    .groupby("strada_sintetica")["Strada"]
    .nunique()
    .sort_values(ascending=False)
)

print("Unique detailed Strada analysed:", len(families_per_road))

print(
    "Strada mapped to exactly one strada_sintetica:",
    (families_per_road == 1).sum()
)

print(
    "Strada mapped to more than one strada_sintetica:",
    (families_per_road > 1).sum()
)

print("\nDistribution of synthetic families per Strada:")
display(
    families_per_road
    .value_counts()
    .sort_index()
    .rename_axis("number_of_synthetic_families")
    .reset_index(name="number_of_detailed_roads")
)

print("\nNumber of detailed roads per strada_sintetica:")
display(roads_per_family.describe())

print("\nLargest road families:")
display(
    roads_per_family
    .head(20)
    .rename("number_of_detailed_roads")
    .reset_index()
)

In [ ]:
import re
import pandas as pd

def normalize_family(value):
    """Normalize an existing strada_sintetica value for comparison."""
    if pd.isna(value):
        return pd.NA

    return re.sub(
        r"[^A-Z0-9]",
        "",
        str(value).upper()
    )


def extract_road_family(strada):
    """
    Extract the main road code from the beginning of Strada.

    Examples:
    SS 106 RACC. JONICA       -> SS106
    SS 7 BIS ...              -> SS7
    A29 ...                   -> A29
    AU A1-D1 ...              -> A1
    RA R.A. 11 ...            -> RA11
    NS NSA 167 ...            -> NSA167
    E E78                     -> E78
    """
    if pd.isna(strada):
        return pd.NA

    text = str(strada).upper().strip()
    text = re.sub(r"\s+", " ", text)

    patterns = [
        # Standard state roads
        (r"^SS\s*(\d+)", lambda m: f"SS{m.group(1)}"),

        # Motorways recorded directly
        (r"^A\s*(\d+)", lambda m: f"A{m.group(1)}"),

        # Motorways preceded by AU
        (r"^AU\s+A\s*(\d+)", lambda m: f"A{m.group(1)}"),

        # Raccordi autostradali
        (r"^RA\s+R\.?\s*A\.?\s*(\d+)", lambda m: f"RA{m.group(1)}"),

        # New ANAS roads
        (r"^NS\s+NSA\s*(\d+)", lambda m: f"NSA{m.group(1)}"),

        # European roads
        (r"^E\s+E\s*(\d+)", lambda m: f"E{m.group(1)}"),
    ]

    for pattern, formatter in patterns:
        match = re.search(pattern, text)

        if match:
            return formatter(match)

    return pd.NA


# Use only rows where both identifiers are available
family_validation = (
    train.loc[
        train["Strada"].notna()
        & train["strada_sintetica"].notna(),
        ["Strada", "strada_sintetica"]
    ]
    .drop_duplicates()
    .copy()
)

family_validation["family_extracted"] = (
    family_validation["Strada"]
    .apply(extract_road_family)
)

family_validation["family_known"] = (
    family_validation["strada_sintetica"]
    .apply(normalize_family)
)

family_validation["correct_match"] = (
    family_validation["family_extracted"]
    == family_validation["family_known"]
)

print("Detailed roads evaluated:", len(family_validation))

print(
    "Family successfully extracted:",
    family_validation["family_extracted"].notna().sum()
)

print(
    "Correct matches:",
    family_validation["correct_match"].sum()
)

print(
    "Accuracy:",
    round(
        family_validation["correct_match"].mean() * 100,
        2
    ),
    "%"
)

print("\nMismatches or unresolved cases:")

display(
    family_validation.loc[
        ~family_validation["correct_match"].fillna(False),
        [
            "Strada",
            "strada_sintetica",
            "family_extracted",
            "family_known"
        ]
    ].head(50)
)

In [ ]:
train["road_family"] = train["Strada"].apply(extract_road_family)
test["road_family"] = test["strada"].apply(extract_road_family)

historical_families = set(
    train.loc[train["Strada"].notna(), "road_family"].dropna()
)

test_families = set(test["road_family"].dropna())

unmatched_test_df = (
    test.loc[
        ~normalize_road(test["strada"]).isin(original_roads),
        ["strada", "road_family"]
    ]
    .drop_duplicates()
    .copy()
)

unmatched_test_df["family_has_historical_data"] = (
    unmatched_test_df["road_family"].isin(historical_families)
)

print("Unmatched test roads:", len(unmatched_test_df))

print(
    "Unmatched roads whose family has historical data:",
    unmatched_test_df["family_has_historical_data"].sum()
)

print(
    "Unmatched roads whose family has no historical data:",
    (~unmatched_test_df["family_has_historical_data"]).sum()
)

display(
    unmatched_test_df[
        ~unmatched_test_df["family_has_historical_data"]
    ].head(50)
)

In [ ]:
from pathlib import Path
import pandas as pd

metadata_path = list(
    Path("/kaggle/input").rglob("road_metadata.csv")
)[0]

road_metadata = pd.read_csv(
    metadata_path,
    sep=";"
)

print("File:", metadata_path)
print("Shape:", road_metadata.shape)

display(road_metadata.head())

In [ ]:
# Normalize the road identifier in the metadata
road_metadata["road_clean"] = normalize_road(road_metadata["strada"])

# One row per road in the test set
test_road_groups = (
    test[["strada", "road_family"]]
    .drop_duplicates()
    .copy()
)

test_road_groups["road_clean"] = normalize_road(
    test_road_groups["strada"]
)

# Classify the forecasting situation
test_road_groups["forecast_group"] = np.select(
    [
        test_road_groups["road_clean"].isin(original_roads),

        ~test_road_groups["road_clean"].isin(original_roads)
        & test_road_groups["road_family"].isin(historical_families)
    ],
    [
        "seen_road",
        "unseen_road_seen_family"
    ],
    default="unseen_family"
)

# Identify whether metadata exists for each road
metadata_roads = set(
    road_metadata["road_clean"].dropna()
)

test_road_groups["metadata_row_exists"] = (
    test_road_groups["road_clean"].isin(metadata_roads)
)

# Merge the numeric metadata
metadata_columns = [
    "n_tratte",
    "comuni_attraversati",
    "province_attraversate",
    "km_min",
    "km_max",
    "consistenza_gg_media",
    "leggeri_media",
    "pesanti_media"
]

test_road_groups = test_road_groups.merge(
    road_metadata[
        ["road_clean"] + metadata_columns
    ],
    on="road_clean",
    how="left",
    validate="one_to_one"
)

# Check whether at least one numeric metadata value is available
test_road_groups["numeric_metadata_available"] = (
    test_road_groups[metadata_columns]
    .notna()
    .any(axis=1)
)

metadata_by_group = (
    test_road_groups
    .groupby("forecast_group")
    .agg(
        number_of_roads=("strada", "size"),
        metadata_row_exists=("metadata_row_exists", "sum"),
        numeric_metadata_available=(
            "numeric_metadata_available",
            "sum"
        )
    )
)

metadata_by_group["numeric_metadata_missing"] = (
    metadata_by_group["number_of_roads"]
    - metadata_by_group["numeric_metadata_available"]
)

metadata_by_group["percentage_with_numeric_metadata"] = (
    100
    * metadata_by_group["numeric_metadata_available"]
    / metadata_by_group["number_of_roads"]
).round(2)

display(metadata_by_group)

In [ ]:
# Temporary historical month variable
train["data_ins_clean"] = pd.to_datetime(
    train["Data Ins"],
    errors="coerce",
    dayfirst=True
)

train["month_clean"] = (
    train["data_ins_clean"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

train["road_clean"] = normalize_road(train["Strada"])

# Keep only roads that appear exactly in the test set
seen_history = train[
    train["road_clean"].isin(test_roads)
    & train["month_clean"].notna()
].copy()

# Historical coverage by road
road_history_summary = (
    seen_history
    .groupby("road_clean")
    .agg(
        raw_records=("road_clean", "size"),
        observed_months=("month_clean", "nunique"),
        first_observed_month=("month_clean", "min"),
        last_observed_month=("month_clean", "max"),
        observed_years=("data_ins_clean", lambda x: x.dt.year.nunique())
    )
    .reset_index()
)

# Total possible months between 2016-01 and 2023-12
total_historical_months = 8 * 12

road_history_summary["percentage_months_observed"] = (
    100
    * road_history_summary["observed_months"]
    / total_historical_months
).round(2)

print("Seen roads analysed:", len(road_history_summary))

display(
    road_history_summary[
        [
            "raw_records",
            "observed_months",
            "observed_years",
            "percentage_months_observed"
        ]
    ].describe()
)

print("\nRoads with the least historical coverage:")

display(
    road_history_summary
    .sort_values(
        ["observed_months", "raw_records"],
        ascending=True
    )
    .head(30)
)

In [ ]:
# Roads expected to have exact historical matches
expected_seen_roads = set(
    test_road_groups.loc[
        test_road_groups["forecast_group"] == "seen_road",
        "road_clean"
    ]
)

# Roads that survived the date parsing and coverage calculation
analysed_seen_roads = set(
    road_history_summary["road_clean"]
)

missing_after_date_parsing = sorted(
    expected_seen_roads - analysed_seen_roads
)

print(
    "Matched roads missing from the historical coverage summary:",
    len(missing_after_date_parsing)
)

missing_date_diagnosis = (
    train.loc[
        train["road_clean"].isin(missing_after_date_parsing),
        ["Strada", "Data Ins", "data_ins_clean", "file_origine"]
    ]
    .copy()
)

display(missing_date_diagnosis.head(50))

print("\nRaw date examples:")
display(
    missing_date_diagnosis["Data Ins"]
    .value_counts(dropna=False)
    .head(30)
)

In [ ]:
train["data_ins_clean"] = pd.to_datetime(
    train["Data Ins"],
    format="%Y-%m-%d",
    errors="coerce"
)

train["month_clean"] = (
    train["data_ins_clean"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

print("Invalid dates:", train["data_ins_clean"].isna().sum())
print("First date:", train["data_ins_clean"].min())
print("Last date:", train["data_ins_clean"].max())

In [ ]:
# Recreate the monthly variable using the corrected dates
train["month_clean"] = (
    train["data_ins_clean"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Keep historical records belonging to roads requested in the test set
seen_history = train[
    train["road_clean"].isin(test_roads)
    & train["month_clean"].notna()
].copy()

road_history_summary = (
    seen_history
    .groupby("road_clean")
    .agg(
        raw_records=("road_clean", "size"),
        observed_months=("month_clean", "nunique"),
        first_observed_month=("month_clean", "min"),
        last_observed_month=("month_clean", "max"),
        observed_years=("data_ins_clean", lambda x: x.dt.year.nunique())
    )
    .reset_index()
)

road_history_summary["percentage_months_observed"] = (
    100 * road_history_summary["observed_months"] / 96
).round(2)

print("Seen roads analysed:", len(road_history_summary))

display(
    road_history_summary[
        [
            "raw_records",
            "observed_months",
            "observed_years",
            "percentage_months_observed"
        ]
    ].describe()
)

In [ ]:
print("Number of columns:", train.shape[1])
print("\nAll columns:")
print(train.columns.tolist())

keywords = [
    "id",
    "case",
    "caso",
    "ticket",
    "richiesta",
    "pratica",
    "protocol",
    "segnalazione",
    "numero",
    "codice"
]

possible_id_columns = [
    column for column in train.columns
    if any(keyword in column.lower() for keyword in keywords)
]

print("\nPossible identifier columns:")
print(possible_id_columns)

if possible_id_columns:
    identifier_audit = pd.DataFrame({
        "data_type": train[possible_id_columns].dtypes.astype(str),
        "non_missing": train[possible_id_columns].notna().sum(),
        "missing": train[possible_id_columns].isna().sum(),
        "unique_values": train[possible_id_columns].nunique(dropna=True)
    })

    identifier_audit["unique_percentage"] = (
        100
        * identifier_audit["unique_values"]
        / identifier_audit["non_missing"]
    ).round(2)

    display(identifier_audit)

In [ ]:
original_columns = [
    "Localita",
    "Provincia",
    "Data Ins",
    "Ora Ins",
    "Canale arrivo",
    "Tipologia",
    "Categoria",
    "Gruppo",
    "Strada",
    "Tipo Persona",
    "file_origine",
    "strada_sintetica",
    "n_tratte",
    "comuni_attraversati",
    "province_attraversate",
    "km_min",
    "km_max",
    "consistenza_gg_media",
    "leggeri_media",
    "pesanti_media",
    "mese_data",
    "anno",
    "mese"
]

exact_duplicate_mask = train.duplicated(
    subset=original_columns,
    keep=False
)

print("Rows belonging to an exact duplicate group:",
      exact_duplicate_mask.sum())

print("Exact duplicate rows beyond the first copy:",
      train.duplicated(
          subset=original_columns,
          keep="first"
      ).sum())

print("Percentage duplicated beyond first copy:",
      round(
          100 * train.duplicated(
              subset=original_columns,
              keep="first"
          ).mean(),
          2
      ),
      "%"
)

display(
    train.loc[exact_duplicate_mask, original_columns]
    .sort_values(original_columns)
    .head(30)
)

In [ ]:
duplicate_beyond_first = train.duplicated(
    subset=original_columns,
    keep="first"
)

duplicate_impact = train.loc[
    duplicate_beyond_first
].copy()

duplicate_impact["has_road"] = duplicate_impact["Strada"].notna()

duplicate_impact["road_in_test"] = (
    duplicate_impact["road_clean"].isin(test_roads)
)

print("Excess exact duplicates:", len(duplicate_impact))
print(
    "Duplicates with Strada:",
    duplicate_impact["has_road"].sum()
)
print(
    "Duplicates associated with a test road:",
    duplicate_impact["road_in_test"].sum()
)

display(
    duplicate_impact[
        duplicate_impact["road_in_test"]
    ][
        ["Strada", "Data Ins", "Ora Ins", "Tipologia", "Categoria"]
    ].head(30)
)

In [ ]:
case_signature_columns = [
    "Localita",
    "Provincia",
    "Data Ins",
    "Ora Ins",
    "Canale arrivo",
    "Tipologia",
    "Categoria",
    "Gruppo",
    "Strada",
    "Tipo Persona",
    "strada_sintetica"
]

signature_summary = (
    train
    .groupby(
        case_signature_columns,
        dropna=False
    )
    .agg(
        number_of_rows=("file_origine", "size"),
        number_of_source_files=("file_origine", "nunique"),
        first_source_file=("file_origine", "min"),
        last_source_file=("file_origine", "max")
    )
    .reset_index()
)

repeated_across_sources = signature_summary[
    signature_summary["number_of_source_files"] > 1
].copy()

print(
    "Candidate cases appearing in multiple source files:",
    len(repeated_across_sources)
)

print(
    "Total rows represented by these candidates:",
    repeated_across_sources["number_of_rows"].sum()
)

print(
    "Potential excess rows beyond one per candidate case:",
    (
        repeated_across_sources["number_of_rows"] - 1
    ).sum()
)

display(
    repeated_across_sources
    .sort_values(
        ["number_of_source_files", "number_of_rows"],
        ascending=False
    )
    .head(30)
)

In [ ]:
train["registration_datetime"] = pd.to_datetime(
    train["Data Ins"].astype(str)
    + " "
    + train["Ora Ins"].astype(str),
    errors="coerce"
)

timestamp_summary = (
    train.groupby("registration_datetime", dropna=False)
    .agg(
        number_of_rows=("registration_datetime", "size"),
        number_of_roads=("Strada", "nunique"),
        number_of_categories=("Categoria", "nunique"),
        number_of_groups=("Gruppo", "nunique")
    )
    .reset_index()
)

print("Valid registration timestamps:",
      train["registration_datetime"].notna().sum())

print("Unique registration timestamps:",
      train["registration_datetime"].nunique())

print(
    "Timestamps appearing more than once:",
    (timestamp_summary["number_of_rows"] > 1).sum()
)

print(
    "Rows belonging to repeated timestamps:",
    timestamp_summary.loc[
        timestamp_summary["number_of_rows"] > 1,
        "number_of_rows"
    ].sum()
)

display(
    timestamp_summary[
        timestamp_summary["number_of_rows"] > 1
    ]
    .sort_values("number_of_rows", ascending=False)
    .head(30)
)

In [ ]:
stable_case_columns = [
    "registration_datetime",
    "Strada",
    "Localita",
    "Provincia",
    "Canale arrivo",
    "Tipo Persona"
]

candidate_case_groups = (
    train.loc[
        train["registration_datetime"].notna()
    ]
    .groupby(
        stable_case_columns,
        dropna=False
    )
    .agg(
        number_of_rows=("registration_datetime", "size"),
        number_of_tipologie=("Tipologia", "nunique"),
        number_of_categorie=("Categoria", "nunique"),
        number_of_gruppi=("Gruppo", "nunique"),
        number_of_source_files=("file_origine", "nunique")
    )
    .reset_index()
)

repeated_candidate_cases = candidate_case_groups[
    candidate_case_groups["number_of_rows"] > 1
].copy()

print(
    "Candidate repeated cases:",
    len(repeated_candidate_cases)
)

print(
    "Rows represented:",
    repeated_candidate_cases["number_of_rows"].sum()
)

print(
    "Potential excess rows:",
    (
        repeated_candidate_cases["number_of_rows"] - 1
    ).sum()
)

display(
    repeated_candidate_cases
    .sort_values("number_of_rows", ascending=False)
    .head(30)
)

In [ ]:
import numpy as np
import pandas as pd
import re

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_log_error


# ---------------------------------------------------------
# 1. Helper functions
# ---------------------------------------------------------

def normalize_road(series):
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .str.replace('""', '"', regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )


def extract_road_family(strada):
    if pd.isna(strada):
        return pd.NA

    text = str(strada).upper().strip()
    text = re.sub(r"\s+", " ", text)

    patterns = [
        (r"^SS\s*(\d+)", lambda m: f"SS{m.group(1)}"),
        (r"^A\s*(\d+)", lambda m: f"A{m.group(1)}"),
        (r"^AU\s+A\s*(\d+)", lambda m: f"A{m.group(1)}"),
        (r"^RA\s+R\.?\s*A\.?\s*(\d+)", lambda m: f"RA{m.group(1)}"),
        (r"^NS\s+NSA\s*(\d+)", lambda m: f"NSA{m.group(1)}"),
        (r"^E\s+E\s*(\d+)", lambda m: f"E{m.group(1)}"),
    ]

    for pattern, formatter in patterns:
        match = re.search(pattern, text)
        if match:
            return formatter(match)

    # Fallback: first token
    return text.split()[0] if text else pd.NA


# ---------------------------------------------------------
# 2. Correct dates and road identifiers
# ---------------------------------------------------------

train["data_ins_clean"] = pd.to_datetime(
    train["Data Ins"],
    format="%Y-%m-%d",
    errors="coerce"
)

train["month_clean"] = (
    train["data_ins_clean"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

train["road_clean"] = normalize_road(train["Strada"])
train["road_family"] = train["Strada"].apply(extract_road_family)

test["road_clean"] = normalize_road(test["strada"])
test["road_family"] = test["strada"].apply(extract_road_family)
test["month_clean"] = pd.to_datetime(test["mese_data"])


# ---------------------------------------------------------
# 3. Remove only exact duplicates
# ---------------------------------------------------------

original_columns = [
    "Localita",
    "Provincia",
    "Data Ins",
    "Ora Ins",
    "Canale arrivo",
    "Tipologia",
    "Categoria",
    "Gruppo",
    "Strada",
    "Tipo Persona",
    "file_origine",
    "strada_sintetica",
    "n_tratte",
    "comuni_attraversati",
    "province_attraversate",
    "km_min",
    "km_max",
    "consistenza_gg_media",
    "leggeri_media",
    "pesanti_media",
    "mese_data",
    "anno",
    "mese"
]

train_deduplicated = train.drop_duplicates(
    subset=original_columns,
    keep="first"
).copy()

print("Original rows:", len(train))
print("Rows after exact deduplication:", len(train_deduplicated))


# ---------------------------------------------------------
# 4. Create observed road-month target
# ---------------------------------------------------------

monthly_target = (
    train_deduplicated.loc[
        train_deduplicated["road_clean"].notna()
        & train_deduplicated["month_clean"].notna()
    ]
    .groupby(
        ["road_clean", "road_family", "month_clean"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "y"})
)

monthly_target["year"] = monthly_target["month_clean"].dt.year
monthly_target["month"] = monthly_target["month_clean"].dt.month

print("Observed road-month targets:", len(monthly_target))
display(monthly_target.head())
display(monthly_target["y"].describe())

In [ ]:
# Normalize metadata road identifier
road_metadata["road_clean"] = normalize_road(
    road_metadata["strada"]
)

metadata_columns = [
    "n_tratte",
    "comuni_attraversati",
    "province_attraversate",
    "km_min",
    "km_max",
    "consistenza_gg_media",
    "leggeri_media",
    "pesanti_media"
]

# Conservative numeric conversion:
# malformed values become missing rather than being guessed
for column in metadata_columns:
    road_metadata[column] = pd.to_numeric(
        road_metadata[column],
        errors="coerce"
    )

road_metadata["road_length_km"] = (
    road_metadata["km_max"] -
    road_metadata["km_min"]
)

road_metadata["traffic_total"] = (
    road_metadata["leggeri_media"] +
    road_metadata["pesanti_media"]
)

road_metadata["heavy_vehicle_share"] = (
    road_metadata["pesanti_media"] /
    road_metadata["traffic_total"].replace(0, np.nan)
)

road_metadata["metadata_available"] = (
    road_metadata[metadata_columns]
    .notna()
    .any(axis=1)
    .astype(int)
)

model_metadata_columns = metadata_columns + [
    "road_length_km",
    "traffic_total",
    "heavy_vehicle_share",
    "metadata_available"
]

road_metadata_model = (
    road_metadata[
        ["road_clean"] + model_metadata_columns
    ]
    .drop_duplicates("road_clean")
)

print(road_metadata_model.shape)
display(road_metadata_model.head())

In [ ]:
def add_historical_features(data, reference):
    """
    Historical target aggregates are calculated only from reference data.

    For validation:
        reference = 2016-2022

    For final test:
        reference = 2016-2023
    """

    result = data.copy()

    road_stats = (
        reference.groupby("road_clean")["y"]
        .agg(
            road_mean="mean",
            road_median="median",
            road_max="max",
            road_observed_months="count"
        )
        .reset_index()
    )

    family_stats = (
        reference.groupby("road_family")["y"]
        .agg(
            family_mean="mean",
            family_median="median",
            family_max="max",
            family_observed_months="count"
        )
        .reset_index()
    )

    calendar_month_stats = (
        reference.groupby("month")["y"]
        .agg(
            calendar_month_mean="mean",
            calendar_month_median="median"
        )
        .reset_index()
    )

    result = result.merge(
        road_stats,
        on="road_clean",
        how="left"
    )

    result = result.merge(
        family_stats,
        on="road_family",
        how="left"
    )

    result = result.merge(
        calendar_month_stats,
        on="month",
        how="left"
    )

    result["global_mean"] = reference["y"].mean()
    result["global_median"] = reference["y"].median()

    result["road_has_history"] = (
        result["road_observed_months"].notna().astype(int)
    )

    result["family_has_history"] = (
        result["family_observed_months"].notna().astype(int)
    )

    return result

In [ ]:
# Temporal split
train_fold = monthly_target[
    monthly_target["year"] <= 2022
].copy()

validation_fold = monthly_target[
    monthly_target["year"] == 2023
].copy()

# Build features without using 2023 targets
train_fold_features = add_historical_features(
    train_fold,
    train_fold
)

validation_fold_features = add_historical_features(
    validation_fold,
    train_fold
)

# Merge road metadata
train_fold_features = train_fold_features.merge(
    road_metadata_model,
    on="road_clean",
    how="left"
)

validation_fold_features = validation_fold_features.merge(
    road_metadata_model,
    on="road_clean",
    how="left"
)

# Road type
train_fold_features["road_type"] = (
    train_fold_features["road_clean"]
    .str.extract(r"^([A-Z]+)", expand=False)
    .fillna("UNKNOWN")
)

validation_fold_features["road_type"] = (
    validation_fold_features["road_clean"]
    .str.extract(r"^([A-Z]+)", expand=False)
    .fillna("UNKNOWN")
)

# Time index
train_fold_features["time_index"] = (
    (train_fold_features["year"] - 2016) * 12
    + train_fold_features["month"]
)

validation_fold_features["time_index"] = (
    (validation_fold_features["year"] - 2016) * 12
    + validation_fold_features["month"]
)

In [ ]:
categorical_features = [
    "road_clean",
    "road_family",
    "road_type",
    "month"
]

numeric_features = [
    "year",
    "time_index",
    "road_mean",
    "road_median",
    "road_max",
    "road_observed_months",
    "family_mean",
    "family_median",
    "family_max",
    "family_observed_months",
    "calendar_month_mean",
    "calendar_month_median",
    "global_mean",
    "global_median",
    "road_has_history",
    "family_has_history"
] + model_metadata_columns

feature_columns = categorical_features + numeric_features

X_train = train_fold_features[feature_columns].copy()
y_train = train_fold_features["y"].copy()

X_valid = validation_fold_features[feature_columns].copy()
y_valid = validation_fold_features["y"].copy()

# CatBoost categorical columns cannot contain nulls
for column in categorical_features:
    X_train[column] = X_train[column].astype("string").fillna("MISSING")
    X_valid[column] = X_valid[column].astype("string").fillna("MISSING")

In [ ]:
catboost_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    l2_leaf_reg=5,
    verbose=100,
    allow_writing_files=False
)

catboost_model.fit(
    X_train,
    np.log1p(y_train),
    cat_features=categorical_features,
    eval_set=(X_valid, np.log1p(y_valid)),
    early_stopping_rounds=100,
    verbose=100
)

validation_prediction_log = catboost_model.predict(X_valid)

validation_prediction = np.clip(
    np.expm1(validation_prediction_log),
    0,
    None
)

validation_rmsle = np.sqrt(
    mean_squared_log_error(
        y_valid,
        validation_prediction
    )
)

print("2023 validation RMSLE:", validation_rmsle)

In [ ]:
# Prepare complete training data
full_train_features = add_historical_features(
    monthly_target,
    monthly_target
)

full_train_features = full_train_features.merge(
    road_metadata_model,
    on="road_clean",
    how="left"
)

full_train_features["road_type"] = (
    full_train_features["road_clean"]
    .str.extract(r"^([A-Z]+)", expand=False)
    .fillna("UNKNOWN")
)

full_train_features["time_index"] = (
    (full_train_features["year"] - 2016) * 12
    + full_train_features["month"]
)


# Prepare test rows in their original submission order
test_features = test.copy()

test_features["year"] = test_features["month_clean"].dt.year
test_features["month"] = test_features["month_clean"].dt.month

test_features = add_historical_features(
    test_features,
    monthly_target
)

test_features = test_features.merge(
    road_metadata_model,
    on="road_clean",
    how="left"
)

test_features["road_type"] = (
    test_features["road_clean"]
    .str.extract(r"^([A-Z]+)", expand=False)
    .fillna("UNKNOWN")
)

test_features["time_index"] = (
    (test_features["year"] - 2016) * 12
    + test_features["month"]
)


X_full = full_train_features[feature_columns].copy()
y_full = full_train_features["y"].copy()

X_test = test_features[feature_columns].copy()

for column in categorical_features:
    X_full[column] = X_full[column].astype("string").fillna("MISSING")
    X_test[column] = X_test[column].astype("string").fillna("MISSING")

In [ ]:
final_catboost_model = CatBoostRegressor(
    iterations=catboost_model.get_best_iteration() + 1,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    l2_leaf_reg=5,
    verbose=100,
    allow_writing_files=False
)

final_catboost_model.fit(
    X_full,
    np.log1p(y_full),
    cat_features=categorical_features,
    verbose=100
)

test_prediction = np.clip(
    np.expm1(
        final_catboost_model.predict(X_test)
    ),
    0,
    None
)

print("Predictions:", len(test_prediction))
print("Minimum:", test_prediction.min())
print("Maximum:", test_prediction.max())
print("Mean:", test_prediction.mean())

assert len(test_prediction) == len(test)
assert len(test_prediction) == 15048

In [ ]:
submission = pd.DataFrame({
    "prediction": test_prediction
})

submission.to_csv(
    "/kaggle/working/submission_catboost.csv",
    index=False,
    header=False
)

display(submission.head(20))

In [ ]:
# Convert predictions to non-negative integer counts
test_prediction_integer = np.rint(
    np.clip(test_prediction, 0, None)
).astype(int)

submission = pd.DataFrame({
    "prediction": test_prediction_integer
})

submission.to_csv(
    "/kaggle/working/submission_catboost.csv",
    index=False,
    header=False
)

print("Number of predictions:", len(submission))
print("Minimum prediction:", submission["prediction"].min())
print("Maximum prediction:", submission["prediction"].max())

display(submission.head(20))

In [ ]:
# RMSLE with continuous predictions
rmsle_continuous = np.sqrt(
    mean_squared_log_error(
        y_valid,
        validation_prediction
    )
)

# RMSLE after integer rounding
validation_prediction_integer = np.rint(
    np.clip(validation_prediction, 0, None)
).astype(int)

rmsle_integer = np.sqrt(
    mean_squared_log_error(
        y_valid,
        validation_prediction_integer
    )
)

print("Continuous RMSLE:", rmsle_continuous)
print("Integer RMSLE:", rmsle_integer)

In [ ]:
import pandas as pd

submission_check = pd.read_csv(
    "/kaggle/working/submission_catboost.csv",
    header=None
)

print(submission_check.shape)
display(submission_check.head())

In [ ]:
import os

file_path = "/kaggle/working/submission_catboost.csv"

print("File exists:", os.path.exists(file_path))
print("File size:", os.path.getsize(file_path) if os.path.exists(file_path) else "Not found")

In [ ]:
import os
from IPython.display import FileLink, display

os.chdir("/kaggle/working")
display(FileLink("submission_catboost.csv"))

In [ ]:
import glob
import os
import pandas as pd

metadata_files = glob.glob(
    "/kaggle/input/datasets/matascerda/new-metadata/road_metadata (1).csv",
    recursive=True
)

metadata_candidates = []

for path in metadata_files:
    try:
        candidate = pd.read_csv(
            path,
            sep=None,
            engine="python"
        )

        metadata_candidates.append(
            {
                "path": path,
                "rows": len(candidate),
                "columns": len(candidate.columns),
                "data": candidate
            }
        )
    except Exception:
        pass

metadata_candidates = sorted(
    metadata_candidates,
    key=lambda x: x["columns"],
    reverse=True
)

for candidate in metadata_candidates:
    print(
        candidate["columns"],
        "columns |",
        candidate["rows"],
        "rows |",
        candidate["path"]
    )

# Select the new metadata file: the one with 42 columns
road_metadata = metadata_candidates[0]["data"].copy()

print("\nSelected metadata shape:", road_metadata.shape)
print("\nColumns:")
print(road_metadata.columns.tolist())

In [ ]:
# Identify the new annual Tipologia columns
tipologia_2024_columns = [
    column for column in road_metadata.columns
    if column.startswith("2024_tipologia_")
]

tipologia_2025_columns = [
    column for column in road_metadata.columns
    if column.startswith("2025_tipologia_")
]

# Convert them safely to numeric
for column in tipologia_2024_columns + tipologia_2025_columns:
    road_metadata[column] = pd.to_numeric(
        road_metadata[column],
        errors="coerce"
    )

# Annual totals by road
road_metadata["annual_total_2024"] = (
    road_metadata[tipologia_2024_columns]
    .sum(axis=1, min_count=1)
)

road_metadata["annual_total_2025"] = (
    road_metadata[tipologia_2025_columns]
    .sum(axis=1, min_count=1)
)

print("2024 Tipologia columns:", len(tipologia_2024_columns))
print("2025 Tipologia columns:", len(tipologia_2025_columns))

print("\nTotal cases across all roads:")
print("2024:", road_metadata["annual_total_2024"].sum())
print("2025:", road_metadata["annual_total_2025"].sum())

print("\nRoads with missing annual total:")
print(
    "2024:",
    road_metadata["annual_total_2024"].isna().sum()
)
print(
    "2025:",
    road_metadata["annual_total_2025"].isna().sum()
)

print("\nRoads with annual total equal to zero:")
print(
    "2024:",
    (road_metadata["annual_total_2024"] == 0).sum()
)
print(
    "2025:",
    (road_metadata["annual_total_2025"] == 0).sum()
)

display(
    road_metadata[
        [
            "strada",
            "annual_total_2024",
            "annual_total_2025"
        ]
    ].head(20)
)

In [ ]:
import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Add raw CatBoost predictions to test rows
# ---------------------------------------------------------

calibrated_test = test_features.copy()

calibrated_test["raw_prediction"] = np.clip(
    test_prediction,
    0,
    None
)

calibrated_test["_original_order"] = np.arange(len(calibrated_test))


# ---------------------------------------------------------
# 2. Prepare annual totals from the new metadata
# ---------------------------------------------------------

road_metadata["road_clean"] = normalize_road(
    road_metadata["strada"]
)

annual_totals = road_metadata[
    [
        "road_clean",
        "annual_total_2024",
        "annual_total_2025"
    ]
].drop_duplicates("road_clean")

calibrated_test = calibrated_test.merge(
    annual_totals,
    on="road_clean",
    how="left"
)

calibrated_test["annual_total"] = np.where(
    calibrated_test["year"] == 2024,
    calibrated_test["annual_total_2024"],
    calibrated_test["annual_total_2025"]
)

assert calibrated_test["annual_total"].notna().all()


# ---------------------------------------------------------
# 3. Convert raw predictions into monthly shares
# ---------------------------------------------------------

group_columns = ["road_clean", "year"]

calibrated_test["raw_annual_sum"] = (
    calibrated_test
    .groupby(group_columns)["raw_prediction"]
    .transform("sum")
)

calibrated_test["months_in_group"] = (
    calibrated_test
    .groupby(group_columns)["month"]
    .transform("count")
)

# Normally use CatBoost monthly proportions.
# If all raw predictions are zero, use an equal 1/12 distribution.
calibrated_test["monthly_share"] = np.where(
    calibrated_test["raw_annual_sum"] > 0,
    calibrated_test["raw_prediction"]
    / calibrated_test["raw_annual_sum"],
    1 / calibrated_test["months_in_group"]
)

calibrated_test["calibrated_continuous"] = (
    calibrated_test["monthly_share"]
    * calibrated_test["annual_total"]
)


# ---------------------------------------------------------
# 4. Convert to integers while preserving exact annual sums
# ---------------------------------------------------------

def integer_allocation(group):
    group = group.copy()

    annual_total = int(round(group["annual_total"].iloc[0]))

    # Initial integer allocation
    group["prediction"] = np.floor(
        group["calibrated_continuous"]
    ).astype(int)

    # Remaining cases that still need to be assigned
    remaining = annual_total - group["prediction"].sum()

    # Assign remaining cases to months with largest decimals
    if remaining > 0:
        fractional_part = (
            group["calibrated_continuous"]
            - np.floor(group["calibrated_continuous"])
        )

        selected_indices = (
            fractional_part
            .sort_values(ascending=False)
            .index[:remaining]
        )

        group.loc[selected_indices, "prediction"] += 1

    return group


calibrated_test = (
    calibrated_test
    .groupby(group_columns, group_keys=False)
    .apply(integer_allocation)
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 5. Restore exact original test order
# ---------------------------------------------------------

calibrated_test = calibrated_test.sort_values(
    "_original_order"
).reset_index(drop=True)


# ---------------------------------------------------------
# 6. Validate annual constraints
# ---------------------------------------------------------

annual_check = (
    calibrated_test
    .groupby(group_columns)
    .agg(
        predicted_annual_total=("prediction", "sum"),
        required_annual_total=("annual_total", "first"),
        number_of_months=("month", "count")
    )
    .reset_index()
)

annual_check["difference"] = (
    annual_check["predicted_annual_total"]
    - annual_check["required_annual_total"]
)

print("Number of predictions:", len(calibrated_test))
print(
    "Road-years with incorrect annual total:",
    (annual_check["difference"] != 0).sum()
)
print(
    "Road-years without 12 months:",
    (annual_check["number_of_months"] != 12).sum()
)
print(
    "Minimum monthly prediction:",
    calibrated_test["prediction"].min()
)
print(
    "Maximum monthly prediction:",
    calibrated_test["prediction"].max()
)

assert len(calibrated_test) == 15048
assert (annual_check["difference"] == 0).all()
assert (annual_check["number_of_months"] == 12).all()
assert (calibrated_test["prediction"] >= 0).all()


# ---------------------------------------------------------
# 7. Save final submission
# ---------------------------------------------------------

submission_calibrated = calibrated_test[
    ["prediction"]
].astype(int)

submission_calibrated.to_csv(
    "/kaggle/working/submission_catboost_annual_calibrated.csv",
    index=False,
    header=False
)

display(submission_calibrated.head(20))

In [ ]:
final_check = pd.read_csv(
    "/kaggle/working/submission_catboost_annual_calibrated.csv",
    header=None
)

print(final_check.shape)
print(final_check.dtypes)
print(final_check.head(20))

In [ ]:
import base64
from IPython.display import HTML, display

file_path = "/kaggle/working/submission_catboost_annual_calibrated.csv"
file_name = "submission_catboost_annual_calibrated.csv"

with open(file_path, "rb") as file:
    file_data = base64.b64encode(file.read()).decode()

download_link = f"""
<a download="{file_name}"
   href="data:text/csv;base64,{file_data}"
   style="
       display:inline-block;
       padding:10px 16px;
       background:#1a73e8;
       color:white;
       text-decoration:none;
       border-radius:5px;
       font-weight:bold;
   ">
   Download {file_name}
</a>
"""

display(HTML(download_link))

In [ ]:
import re
import unicodedata
import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Normalize Tipologia names to match metadata columns
# ---------------------------------------------------------

def normalize_tipologia(value):
    if pd.isna(value) or str(value).strip() == "":
        return "missing"

    text = unicodedata.normalize(
        "NFKD",
        str(value)
    ).encode(
        "ascii",
        "ignore"
    ).decode("ascii")

    text = text.lower().strip()

    # Replace every non-alphanumeric character separately.
    # This preserves patterns such as:
    # "Sala Operativa Nazionale - Sollecito"
    # → "sala_operativa_nazionale___sollecito"
    text = re.sub(r"[^a-z0-9]", "_", text)

    return text


# ---------------------------------------------------------
# 2. Prepare historical incident rows
# ---------------------------------------------------------

historical_tipologia = train_deduplicated.loc[
    train_deduplicated["road_clean"].notna()
    & train_deduplicated["data_ins_clean"].notna()
].copy()

historical_tipologia["year"] = (
    historical_tipologia["data_ins_clean"].dt.year
)

historical_tipologia["road_family"] = (
    historical_tipologia["road_family"]
    .astype("string")
    .fillna("UNKNOWN_FAMILY")
)

historical_tipologia["tipologia_key"] = (
    historical_tipologia["Tipologia"]
    .apply(normalize_tipologia)
)


# ---------------------------------------------------------
# 3. Count each Tipologia by road and year
# ---------------------------------------------------------

annual_tipologia_long = (
    historical_tipologia
    .groupby(
        [
            "road_clean",
            "road_family",
            "year",
            "tipologia_key"
        ],
        as_index=False
    )
    .size()
    .rename(columns={"size": "tipologia_count"})
)

annual_tipologia_wide = (
    annual_tipologia_long
    .pivot_table(
        index=["road_clean", "road_family", "year"],
        columns="tipologia_key",
        values="tipologia_count",
        fill_value=0,
        aggfunc="sum"
    )
    .reset_index()
)

annual_tipologia_wide.columns.name = None

annual_tipologia_wide = annual_tipologia_wide.rename(
    columns={
        column: f"annual_tipologia_{column}"
        for column in annual_tipologia_wide.columns
        if column not in ["road_clean", "road_family", "year"]
    }
)

historical_tipologia_columns = [
    column
    for column in annual_tipologia_wide.columns
    if column.startswith("annual_tipologia_")
]

annual_tipologia_wide["annual_total"] = (
    annual_tipologia_wide[
        historical_tipologia_columns
    ].sum(axis=1)
)


# ---------------------------------------------------------
# 4. Verify that no incidents were lost
# ---------------------------------------------------------

direct_annual_counts = (
    historical_tipologia
    .groupby(["road_clean", "year"])
    .size()
    .reset_index(name="direct_annual_total")
)

annual_check = annual_tipologia_wide.merge(
    direct_annual_counts,
    on=["road_clean", "year"],
    how="left"
)

annual_check["difference"] = (
    annual_check["annual_total"]
    - annual_check["direct_annual_total"]
)


# ---------------------------------------------------------
# 5. Compare historical categories with new metadata
# ---------------------------------------------------------

metadata_tipologia_keys = sorted({
    column.split("_tipologia_", 1)[1]
    for column in road_metadata.columns
    if "_tipologia_" in column
})

historical_tipologia_keys = sorted(
    historical_tipologia["tipologia_key"].unique()
)

historical_not_in_metadata = sorted(
    set(historical_tipologia_keys)
    - set(metadata_tipologia_keys)
)

metadata_not_in_history = sorted(
    set(metadata_tipologia_keys)
    - set(historical_tipologia_keys)
)


# ---------------------------------------------------------
# 6. Results
# ---------------------------------------------------------

print(
    "Historical road-year observations:",
    len(annual_tipologia_wide)
)

print(
    "Historical Tipologia variables:",
    len(historical_tipologia_columns)
)

print(
    "Road-years with inconsistent annual totals:",
    (annual_check["difference"] != 0).sum()
)

print("\nTipologia categories in metadata:")
print(metadata_tipologia_keys)

print("\nHistorical categories not found in new metadata:")
print(historical_not_in_metadata)

print("\nMetadata categories not found historically:")
print(metadata_not_in_history)

display(
    annual_tipologia_wide.head()
)

In [ ]:
# ---------------------------------------------------------
# Harmonize historical Tipologia features with new metadata
# ---------------------------------------------------------

all_tipologia_keys = sorted(
    set(metadata_tipologia_keys)
    | set(historical_tipologia_keys)
)

all_annual_tipologia_columns = [
    f"annual_tipologia_{key}"
    for key in all_tipologia_keys
]

# Add future-only categories to historical data as zeros
for column in all_annual_tipologia_columns:
    if column not in annual_tipologia_wide.columns:
        annual_tipologia_wide[column] = 0

# Ensure all category columns are numeric
for column in all_annual_tipologia_columns:
    annual_tipologia_wide[column] = pd.to_numeric(
        annual_tipologia_wide[column],
        errors="coerce"
    ).fillna(0)

# Recalculate annual total using all aligned categories
annual_tipologia_wide["annual_total"] = (
    annual_tipologia_wide[
        all_annual_tipologia_columns
    ].sum(axis=1)
)

# Create annual composition proportions
annual_tipologia_share_columns = []

for column in all_annual_tipologia_columns:
    share_column = column.replace(
        "annual_tipologia_",
        "annual_share_"
    )

    annual_tipologia_wide[share_column] = np.where(
        annual_tipologia_wide["annual_total"] > 0,
        annual_tipologia_wide[column]
        / annual_tipologia_wide["annual_total"],
        0
    )

    annual_tipologia_share_columns.append(share_column)

# Validation checks
share_sum = annual_tipologia_wide[
    annual_tipologia_share_columns
].sum(axis=1)

print("Aligned Tipologia categories:", len(all_tipologia_keys))
print(all_tipologia_keys)

print(
    "\nRoad-years with positive total:",
    (annual_tipologia_wide["annual_total"] > 0).sum()
)

print(
    "Positive-total road-years whose shares do not sum to 1:",
    (
        (annual_tipologia_wide["annual_total"] > 0)
        & ~np.isclose(share_sum, 1)
    ).sum()
)

print("\nValues of future-only historical variables:")
print(
    annual_tipologia_wide[
        [
            "annual_tipologia_missing",
            "annual_tipologia_pec"
        ]
    ].sum()
)

display(
    annual_tipologia_wide[
        [
            "road_clean",
            "road_family",
            "year",
            "annual_total"
        ]
        + all_annual_tipologia_columns[:5]
        + annual_tipologia_share_columns[:5]
    ].head()
)

In [ ]:
# ---------------------------------------------------------
# Build monthly training data with annual Tipologia features
# ---------------------------------------------------------

# Recreate monthly target if necessary
monthly_target_tipologia = (
    train_deduplicated.loc[
        train_deduplicated["road_clean"].notna()
        & train_deduplicated["data_ins_clean"].notna()
    ]
    .assign(
        month_clean=lambda df:
            df["data_ins_clean"].dt.to_period("M").dt.to_timestamp()
    )
    .groupby(
        ["road_clean", "road_family", "month_clean"],
        as_index=False,
        dropna=False
    )
    .size()
    .rename(columns={"size": "y"})
)

monthly_target_tipologia["year"] = (
    monthly_target_tipologia["month_clean"].dt.year
)

monthly_target_tipologia["month"] = (
    monthly_target_tipologia["month_clean"].dt.month
)

# Merge annual Tipologia composition for the same road-year
monthly_model_data = monthly_target_tipologia.merge(
    annual_tipologia_wide,
    on=["road_clean", "road_family", "year"],
    how="left",
    validate="many_to_one"
)

# Monthly share of the known annual total
monthly_model_data["monthly_share"] = (
    monthly_model_data["y"]
    / monthly_model_data["annual_total"]
)

# Checks
print("Monthly observations:", len(monthly_model_data))

print(
    "Rows without annual Tipologia information:",
    monthly_model_data["annual_total"].isna().sum()
)

print(
    "Rows with invalid monthly share:",
    (
        monthly_model_data["monthly_share"].isna()
        | ~np.isfinite(monthly_model_data["monthly_share"])
    ).sum()
)

print(
    "Minimum monthly share:",
    monthly_model_data["monthly_share"].min()
)

print(
    "Maximum monthly share:",
    monthly_model_data["monthly_share"].max()
)

road_year_share_check = (
    monthly_model_data
    .groupby(["road_clean", "year"])["monthly_share"]
    .sum()
)

print(
    "Road-years whose observed monthly shares do not sum to 1:",
    (~np.isclose(road_year_share_check, 1)).sum()
)

display(
    monthly_model_data[
        [
            "road_clean",
            "road_family",
            "year",
            "month",
            "y",
            "annual_total",
            "monthly_share"
        ]
        + annual_tipologia_share_columns[:5]
    ].head(20)
)

In [ ]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_log_error


# =========================================================
# 1. Prepare physical road metadata
# =========================================================

base_metadata_columns = [
    "n_tratte",
    "comuni_attraversati",
    "province_attraversate",
    "km_min",
    "km_max",
    "consistenza_gg_media",
    "leggeri_media",
    "pesanti_media"
]

road_metadata["road_clean"] = normalize_road(
    road_metadata["strada"]
)

for column in base_metadata_columns:
    road_metadata[column] = pd.to_numeric(
        road_metadata[column],
        errors="coerce"
    )

road_metadata["road_length_km"] = (
    road_metadata["km_max"] -
    road_metadata["km_min"]
)

road_metadata["traffic_total"] = (
    road_metadata["leggeri_media"] +
    road_metadata["pesanti_media"]
)

road_metadata["heavy_vehicle_share"] = (
    road_metadata["pesanti_media"]
    / road_metadata["traffic_total"].replace(0, np.nan)
)

road_metadata["metadata_available"] = (
    road_metadata[base_metadata_columns]
    .notna()
    .any(axis=1)
    .astype(int)
)

model_metadata_columns = base_metadata_columns + [
    "road_length_km",
    "traffic_total",
    "heavy_vehicle_share",
    "metadata_available"
]

road_metadata_model = (
    road_metadata[
        ["road_clean"] + model_metadata_columns
    ]
    .drop_duplicates("road_clean")
)


# =========================================================
# 2. Training observations: 2016–2022
# =========================================================

model_b_train = monthly_model_data[
    monthly_model_data["year"] <= 2022
].copy()


# =========================================================
# 3. Validation grid: every month of 2023
# =========================================================

validation_annual = annual_tipologia_wide[
    annual_tipologia_wide["year"] == 2023
].copy()

validation_months = pd.DataFrame({
    "month": np.arange(1, 13)
})

validation_annual["_key"] = 1
validation_months["_key"] = 1

model_b_valid = (
    validation_annual
    .merge(validation_months, on="_key")
    .drop(columns="_key")
)

# Attach observed 2023 monthly targets.
# Unobserved historical months remain missing, not zero.
validation_targets = monthly_target_tipologia[
    monthly_target_tipologia["year"] == 2023
][
    ["road_clean", "road_family", "year", "month", "y"]
]

model_b_valid = model_b_valid.merge(
    validation_targets,
    on=["road_clean", "road_family", "year", "month"],
    how="left"
)


# =========================================================
# 4. Add metadata and common features
# =========================================================

def enrich_model_b(data):
    result = data.copy()

    result = result.merge(
        road_metadata_model,
        on="road_clean",
        how="left"
    )

    result["road_type"] = (
        result["road_clean"]
        .str.extract(r"^([A-Z]+)", expand=False)
        .fillna("UNKNOWN")
    )

    result["time_index"] = (
        (result["year"] - 2016) * 12
        + result["month"]
    )

    return result


model_b_train = enrich_model_b(model_b_train)
model_b_valid = enrich_model_b(model_b_valid)


# =========================================================
# 5. Define predictors
# =========================================================

categorical_features_b = [
    "road_clean",
    "road_family",
    "road_type",
    "month"
]

numeric_features_b = [
    "year",
    "time_index",
    "annual_total"
] + all_annual_tipologia_columns \
  + annual_tipologia_share_columns \
  + model_metadata_columns

feature_columns_b = (
    categorical_features_b
    + numeric_features_b
)

X_train_b = model_b_train[
    feature_columns_b
].copy()

y_train_b = model_b_train["y"].copy()

# Only observed validation targets are used for early stopping
valid_observed_mask = model_b_valid["y"].notna()

X_valid_observed_b = model_b_valid.loc[
    valid_observed_mask,
    feature_columns_b
].copy()

y_valid_observed_b = model_b_valid.loc[
    valid_observed_mask,
    "y"
].copy()

X_valid_full_b = model_b_valid[
    feature_columns_b
].copy()

for column in categorical_features_b:
    X_train_b[column] = (
        X_train_b[column]
        .astype("string")
        .fillna("MISSING")
    )

    X_valid_observed_b[column] = (
        X_valid_observed_b[column]
        .astype("string")
        .fillna("MISSING")
    )

    X_valid_full_b[column] = (
        X_valid_full_b[column]
        .astype("string")
        .fillna("MISSING")
    )


# =========================================================
# 6. Train CatBoost Model B
# =========================================================

catboost_tipologia = CatBoostRegressor(
    iterations=1200,
    learning_rate=0.04,
    depth=8,
    loss_function="RMSE",
    eval_metric="RMSE",
    l2_leaf_reg=8,
    random_seed=42,
    allow_writing_files=False,
    verbose=100
)

catboost_tipologia.fit(
    X_train_b,
    np.log1p(y_train_b),
    cat_features=categorical_features_b,
    eval_set=(
        X_valid_observed_b,
        np.log1p(y_valid_observed_b)
    ),
    early_stopping_rounds=120,
    verbose=100
)


# =========================================================
# 7. Predict all 12 months and calibrate annual totals
# =========================================================

model_b_valid["raw_prediction"] = np.clip(
    np.expm1(
        catboost_tipologia.predict(X_valid_full_b)
    ),
    0,
    None
)

model_b_valid["raw_annual_sum"] = (
    model_b_valid
    .groupby(["road_clean", "year"])["raw_prediction"]
    .transform("sum")
)

model_b_valid["predicted_share"] = np.where(
    model_b_valid["raw_annual_sum"] > 0,
    model_b_valid["raw_prediction"]
    / model_b_valid["raw_annual_sum"],
    1 / 12
)

model_b_valid["calibrated_prediction"] = (
    model_b_valid["predicted_share"]
    * model_b_valid["annual_total"]
)


# =========================================================
# 8. Evaluate only historically observed 2023 months
# =========================================================

validation_scored_b = model_b_valid[
    model_b_valid["y"].notna()
].copy()

rmsle_model_b = np.sqrt(
    mean_squared_log_error(
        validation_scored_b["y"],
        validation_scored_b["calibrated_prediction"]
    )
)

rounded_prediction_b = np.rint(
    validation_scored_b["calibrated_prediction"]
).astype(int)

rmsle_model_b_rounded = np.sqrt(
    mean_squared_log_error(
        validation_scored_b["y"],
        rounded_prediction_b
    )
)

print("Training observations:", len(model_b_train))
print("Full 2023 validation grid:", len(model_b_valid))
print("Observed 2023 validation targets:", len(validation_scored_b))
print("Best iteration:", catboost_tipologia.get_best_iteration())

print("\nModel B continuous RMSLE:", rmsle_model_b)
print("Model B rounded RMSLE:", rmsle_model_b_rounded)

In [ ]:
# =========================================================
# FINAL MODEL B: CatBoost + annual Tipologia composition
# =========================================================

# ---------------------------------------------------------
# 1. Train using all historical monthly observations
# ---------------------------------------------------------

model_b_full_train = enrich_model_b(
    monthly_model_data.copy()
)

X_full_b = model_b_full_train[
    feature_columns_b
].copy()

y_full_b = model_b_full_train["y"].copy()

for column in categorical_features_b:
    X_full_b[column] = (
        X_full_b[column]
        .astype("string")
        .fillna("MISSING")
    )

best_iterations_b = (
    catboost_tipologia.get_best_iteration() + 1
)

final_catboost_tipologia = CatBoostRegressor(
    iterations=best_iterations_b,
    learning_rate=0.04,
    depth=8,
    loss_function="RMSE",
    l2_leaf_reg=8,
    random_seed=42,
    allow_writing_files=False,
    verbose=100
)

final_catboost_tipologia.fit(
    X_full_b,
    np.log1p(y_full_b),
    cat_features=categorical_features_b,
    verbose=100
)


# ---------------------------------------------------------
# 2. Convert 2024–2025 metadata to the historical structure
# ---------------------------------------------------------

future_annual_frames = []

for forecast_year in [2024, 2025]:

    future_year = road_metadata[
        ["road_clean"]
    ].copy()

    for key in all_tipologia_keys:

        source_column = (
            f"{forecast_year}_tipologia_{key}"
        )

        target_column = (
            f"annual_tipologia_{key}"
        )

        if source_column in road_metadata.columns:
            future_year[target_column] = pd.to_numeric(
                road_metadata[source_column],
                errors="coerce"
            ).fillna(0)
        else:
            future_year[target_column] = 0

    future_year["year"] = forecast_year

    future_year["annual_total"] = (
        future_year[
            all_annual_tipologia_columns
        ].sum(axis=1)
    )

    for count_column, share_column in zip(
        all_annual_tipologia_columns,
        annual_tipologia_share_columns
    ):
        future_year[share_column] = np.where(
            future_year["annual_total"] > 0,
            future_year[count_column]
            / future_year["annual_total"],
            0
        )

    future_annual_frames.append(future_year)

future_annual_tipologia = pd.concat(
    future_annual_frames,
    ignore_index=True
)


# ---------------------------------------------------------
# 3. Prepare the 15,048 test rows
# ---------------------------------------------------------

model_b_test = test.copy()

model_b_test["_original_order"] = np.arange(
    len(model_b_test)
)

model_b_test["road_clean"] = normalize_road(
    model_b_test["strada"]
)

model_b_test["road_family"] = (
    model_b_test["strada"]
    .apply(extract_road_family)
)

model_b_test["month_clean"] = pd.to_datetime(
    model_b_test["mese_data"]
)

model_b_test["year"] = (
    model_b_test["month_clean"].dt.year
)

model_b_test["month"] = (
    model_b_test["month_clean"].dt.month
)

model_b_test = model_b_test.merge(
    future_annual_tipologia,
    on=["road_clean", "year"],
    how="left",
    validate="many_to_one"
)

model_b_test = enrich_model_b(model_b_test)

assert model_b_test["annual_total"].notna().all()


# ---------------------------------------------------------
# 4. Predict raw monthly values
# ---------------------------------------------------------

X_test_b = model_b_test[
    feature_columns_b
].copy()

for column in categorical_features_b:
    X_test_b[column] = (
        X_test_b[column]
        .astype("string")
        .fillna("MISSING")
    )

model_b_test["raw_prediction"] = np.clip(
    np.expm1(
        final_catboost_tipologia.predict(X_test_b)
    ),
    0,
    None
)


# ---------------------------------------------------------
# 5. Convert raw predictions to monthly proportions
# ---------------------------------------------------------

model_b_test["raw_annual_sum"] = (
    model_b_test
    .groupby(
        ["road_clean", "year"]
    )["raw_prediction"]
    .transform("sum")
)

model_b_test["predicted_share"] = np.where(
    model_b_test["raw_annual_sum"] > 0,
    model_b_test["raw_prediction"]
    / model_b_test["raw_annual_sum"],
    1 / 12
)

model_b_test["continuous_prediction"] = (
    model_b_test["predicted_share"]
    * model_b_test["annual_total"]
)


# ---------------------------------------------------------
# 6. Largest-remainder integer allocation
# ---------------------------------------------------------

model_b_test["prediction"] = np.floor(
    model_b_test["continuous_prediction"]
).astype(int)

for _, group_indices in model_b_test.groupby(
    ["road_clean", "year"]
).groups.items():

    group_indices = list(group_indices)

    required_total = int(
        round(
            model_b_test.loc[
                group_indices,
                "annual_total"
            ].iloc[0]
        )
    )

    current_total = int(
        model_b_test.loc[
            group_indices,
            "prediction"
        ].sum()
    )

    remaining = required_total - current_total

    if remaining > 0:

        fractional_parts = (
            model_b_test.loc[
                group_indices,
                "continuous_prediction"
            ]
            - model_b_test.loc[
                group_indices,
                "prediction"
            ]
        )

        selected_indices = (
            fractional_parts
            .sort_values(ascending=False)
            .index[:remaining]
        )

        model_b_test.loc[
            selected_indices,
            "prediction"
        ] += 1


# ---------------------------------------------------------
# 7. Restore original order and validate
# ---------------------------------------------------------

model_b_test = model_b_test.sort_values(
    "_original_order"
).reset_index(drop=True)

annual_check_b = (
    model_b_test
    .groupby(["road_clean", "year"])
    .agg(
        predicted_total=("prediction", "sum"),
        required_total=("annual_total", "first"),
        months=("month", "count")
    )
    .reset_index()
)

annual_check_b["difference"] = (
    annual_check_b["predicted_total"]
    - annual_check_b["required_total"]
)

print("Best iterations:", best_iterations_b)
print("Predictions:", len(model_b_test))
print(
    "Incorrect annual totals:",
    (annual_check_b["difference"] != 0).sum()
)
print(
    "Road-years without 12 months:",
    (annual_check_b["months"] != 12).sum()
)
print(
    "Minimum prediction:",
    model_b_test["prediction"].min()
)
print(
    "Maximum prediction:",
    model_b_test["prediction"].max()
)

assert len(model_b_test) == 15048
assert (annual_check_b["difference"] == 0).all()
assert (annual_check_b["months"] == 12).all()
assert (model_b_test["prediction"] >= 0).all()


# ---------------------------------------------------------
# 8. Save Model B submission
# ---------------------------------------------------------

submission_model_b = model_b_test[
    ["prediction"]
].astype(int)

submission_model_b.to_csv(
    "/kaggle/working/submission_catboost_tipologia.csv",
    index=False,
    header=False
)

display(submission_model_b.head(20))

In [ ]:
# =========================================================
# MODEL C: Tipologia seasonal decomposition
# Validation year: 2023
# Training years: 2016–2022
# =========================================================

# ---------------------------------------------------------
# 1. Prepare historical incident-level data
# ---------------------------------------------------------

seasonality_data = train_deduplicated.loc[
    train_deduplicated["road_clean"].notna()
    & train_deduplicated["data_ins_clean"].notna()
].copy()

seasonality_data["year"] = (
    seasonality_data["data_ins_clean"].dt.year
)

seasonality_data["month"] = (
    seasonality_data["data_ins_clean"].dt.month
)

seasonality_data["tipologia_key"] = (
    seasonality_data["Tipologia"]
    .apply(normalize_tipologia)
)


# ---------------------------------------------------------
# 2. Learn monthly distribution by Tipologia using 2016–2022
# ---------------------------------------------------------

seasonality_train = seasonality_data[
    seasonality_data["year"] <= 2022
].copy()

tipologia_month_counts = (
    seasonality_train
    .groupby(
        ["tipologia_key", "month"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "count"})
)

# Complete missing Tipologia-month combinations with zero
tipologia_grid = pd.MultiIndex.from_product(
    [
        all_tipologia_keys,
        np.arange(1, 13)
    ],
    names=["tipologia_key", "month"]
).to_frame(index=False)

tipologia_month_counts = tipologia_grid.merge(
    tipologia_month_counts,
    on=["tipologia_key", "month"],
    how="left"
)

tipologia_month_counts["count"] = (
    tipologia_month_counts["count"]
    .fillna(0)
)

# Add-one smoothing so no category assigns zero probability
tipologia_month_counts["smoothed_count"] = (
    tipologia_month_counts["count"] + 1
)

tipologia_month_counts["monthly_share"] = (
    tipologia_month_counts["smoothed_count"]
    / tipologia_month_counts
      .groupby("tipologia_key")["smoothed_count"]
      .transform("sum")
)


# ---------------------------------------------------------
# 3. Obtain 2023 annual Tipologia totals
# ---------------------------------------------------------

validation_annual_c = annual_tipologia_wide[
    annual_tipologia_wide["year"] == 2023
].copy()

validation_months_c = pd.DataFrame({
    "month": np.arange(1, 13)
})

validation_annual_c["_key"] = 1
validation_months_c["_key"] = 1

model_c_valid = (
    validation_annual_c
    .merge(validation_months_c, on="_key")
    .drop(columns="_key")
)


# ---------------------------------------------------------
# 4. Predict monthly counts from Tipologia composition
# ---------------------------------------------------------

model_c_valid["continuous_prediction"] = 0.0

for key in all_tipologia_keys:

    annual_column = f"annual_tipologia_{key}"

    category_profile = (
        tipologia_month_counts.loc[
            tipologia_month_counts["tipologia_key"] == key,
            ["month", "monthly_share"]
        ]
        .rename(
            columns={
                "monthly_share": f"share_{key}"
            }
        )
    )

    model_c_valid = model_c_valid.merge(
        category_profile,
        on="month",
        how="left"
    )

    model_c_valid["continuous_prediction"] += (
        model_c_valid[annual_column]
        * model_c_valid[f"share_{key}"]
    )


# ---------------------------------------------------------
# 5. Attach observed 2023 monthly targets
# ---------------------------------------------------------

validation_targets_c = monthly_target_tipologia[
    monthly_target_tipologia["year"] == 2023
][
    ["road_clean", "year", "month", "y"]
]

model_c_valid = model_c_valid.merge(
    validation_targets_c,
    on=["road_clean", "year", "month"],
    how="left"
)


# ---------------------------------------------------------
# 6. Evaluate on observed 2023 road-months
# ---------------------------------------------------------

model_c_scored = model_c_valid[
    model_c_valid["y"].notna()
].copy()

rmsle_model_c = np.sqrt(
    mean_squared_log_error(
        model_c_scored["y"],
        model_c_scored["continuous_prediction"]
    )
)

rounded_model_c = np.rint(
    model_c_scored["continuous_prediction"]
).astype(int)

rmsle_model_c_rounded = np.sqrt(
    mean_squared_log_error(
        model_c_scored["y"],
        rounded_model_c
    )
)

print("Tipologia profiles:", len(all_tipologia_keys))
print("Full 2023 validation grid:", len(model_c_valid))
print("Observed validation targets:", len(model_c_scored))

print("\nModel C continuous RMSLE:", rmsle_model_c)
print("Model C rounded RMSLE:", rmsle_model_c_rounded)

display(
    tipologia_month_counts.head(24)
)

In [ ]:
# =========================================================
# Blend Model B and Model C on 2023 validation
# =========================================================

blend_validation = (
    model_b_valid[
        [
            "road_clean",
            "year",
            "month",
            "y",
            "calibrated_prediction"
        ]
    ]
    .rename(
        columns={
            "calibrated_prediction": "prediction_b"
        }
    )
    .merge(
        model_c_valid[
            [
                "road_clean",
                "year",
                "month",
                "continuous_prediction"
            ]
        ].rename(
            columns={
                "continuous_prediction": "prediction_c"
            }
        ),
        on=["road_clean", "year", "month"],
        how="inner",
        validate="one_to_one"
    )
)

# Score only observed historical months
blend_validation = blend_validation[
    blend_validation["y"].notna()
].copy()

blend_results = []

for weight_b in np.arange(0, 1.01, 0.05):

    weight_c = 1 - weight_b

    blended_prediction = (
        weight_b * blend_validation["prediction_b"]
        + weight_c * blend_validation["prediction_c"]
    )

    blended_prediction = np.clip(
        blended_prediction,
        0,
        None
    )

    rmsle = np.sqrt(
        mean_squared_log_error(
            blend_validation["y"],
            blended_prediction
        )
    )

    blend_results.append({
        "weight_model_b": round(weight_b, 2),
        "weight_model_c": round(weight_c, 2),
        "rmsle": rmsle
    })

blend_results = pd.DataFrame(
    blend_results
).sort_values("rmsle")

display(blend_results.head(10))

best_blend = blend_results.iloc[0]

print(
    "\nBest Model B weight:",
    best_blend["weight_model_b"]
)

print(
    "Best Model C weight:",
    best_blend["weight_model_c"]
)

print(
    "Best blended RMSLE:",
    best_blend["rmsle"]
)

In [ ]:
# =========================================================
# FINAL MODEL C + 75% B / 25% C ENSEMBLE
# =========================================================

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Learn Tipologia seasonality using all 2016–2023 data
# ---------------------------------------------------------

tipologia_month_counts_full = (
    seasonality_data
    .groupby(
        ["tipologia_key", "month"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "count"})
)

tipologia_grid_full = pd.MultiIndex.from_product(
    [
        all_tipologia_keys,
        np.arange(1, 13)
    ],
    names=["tipologia_key", "month"]
).to_frame(index=False)

tipologia_month_counts_full = (
    tipologia_grid_full
    .merge(
        tipologia_month_counts_full,
        on=["tipologia_key", "month"],
        how="left"
    )
)

tipologia_month_counts_full["count"] = (
    tipologia_month_counts_full["count"]
    .fillna(0)
)

# Add-one smoothing
tipologia_month_counts_full["smoothed_count"] = (
    tipologia_month_counts_full["count"] + 1
)

tipologia_month_counts_full["monthly_share"] = (
    tipologia_month_counts_full["smoothed_count"]
    / tipologia_month_counts_full
      .groupby("tipologia_key")["smoothed_count"]
      .transform("sum")
)


# ---------------------------------------------------------
# 2. Prepare Model C test grid
# ---------------------------------------------------------

model_c_test = test.copy()

model_c_test["_original_order"] = np.arange(
    len(model_c_test)
)

model_c_test["road_clean"] = normalize_road(
    model_c_test["strada"]
)

model_c_test["road_family"] = (
    model_c_test["strada"]
    .apply(extract_road_family)
)

model_c_test["month_clean"] = pd.to_datetime(
    model_c_test["mese_data"]
)

model_c_test["year"] = (
    model_c_test["month_clean"].dt.year
)

model_c_test["month"] = (
    model_c_test["month_clean"].dt.month
)

model_c_test = model_c_test.merge(
    future_annual_tipologia,
    on=["road_clean", "year"],
    how="left",
    validate="many_to_one"
)

assert model_c_test["annual_total"].notna().all()


# ---------------------------------------------------------
# 3. Model C continuous predictions
# ---------------------------------------------------------

model_c_test["prediction_c_continuous"] = 0.0

for key in all_tipologia_keys:

    annual_column = f"annual_tipologia_{key}"

    profile = (
        tipologia_month_counts_full.loc[
            tipologia_month_counts_full["tipologia_key"] == key,
            ["month", "monthly_share"]
        ]
        .rename(
            columns={
                "monthly_share": f"profile_{key}"
            }
        )
    )

    model_c_test = model_c_test.merge(
        profile,
        on="month",
        how="left"
    )

    model_c_test["prediction_c_continuous"] += (
        model_c_test[annual_column]
        * model_c_test[f"profile_{key}"]
    )


# Model C monthly shares
model_c_test["share_c"] = np.where(
    model_c_test["annual_total"] > 0,
    model_c_test["prediction_c_continuous"]
    / model_c_test["annual_total"],
    1 / 12
)


# ---------------------------------------------------------
# 4. Prepare Model B shares
# ---------------------------------------------------------

model_b_ensemble = model_b_test[
    [
        "_original_order",
        "road_clean",
        "year",
        "month",
        "annual_total",
        "predicted_share"
    ]
].copy()

model_b_ensemble = model_b_ensemble.rename(
    columns={
        "predicted_share": "share_b"
    }
)


# ---------------------------------------------------------
# 5. Merge Model B and Model C
# ---------------------------------------------------------

ensemble_test = model_b_ensemble.merge(
    model_c_test[
        [
            "_original_order",
            "road_clean",
            "year",
            "month",
            "share_c"
        ]
    ],
    on=[
        "_original_order",
        "road_clean",
        "year",
        "month"
    ],
    how="inner",
    validate="one_to_one"
)

assert len(ensemble_test) == 15048


# ---------------------------------------------------------
# 6. Blend monthly shares
# ---------------------------------------------------------

weight_b = 0.75
weight_c = 0.25

ensemble_test["ensemble_share"] = (
    weight_b * ensemble_test["share_b"]
    + weight_c * ensemble_test["share_c"]
)

# Numerical safeguard: force shares to sum exactly to 1
ensemble_test["share_sum"] = (
    ensemble_test
    .groupby(
        ["road_clean", "year"]
    )["ensemble_share"]
    .transform("sum")
)

ensemble_test["ensemble_share"] = np.where(
    ensemble_test["share_sum"] > 0,
    ensemble_test["ensemble_share"]
    / ensemble_test["share_sum"],
    1 / 12
)

ensemble_test["continuous_prediction"] = (
    ensemble_test["ensemble_share"]
    * ensemble_test["annual_total"]
)


# ---------------------------------------------------------
# 7. Convert to integers preserving annual totals
# ---------------------------------------------------------

ensemble_test["prediction"] = np.floor(
    ensemble_test["continuous_prediction"]
).astype(int)

for _, indices in ensemble_test.groupby(
    ["road_clean", "year"]
).groups.items():

    indices = list(indices)

    annual_total = int(
        round(
            ensemble_test.loc[
                indices,
                "annual_total"
            ].iloc[0]
        )
    )

    current_total = int(
        ensemble_test.loc[
            indices,
            "prediction"
        ].sum()
    )

    remaining = annual_total - current_total

    if remaining > 0:

        fractional_parts = (
            ensemble_test.loc[
                indices,
                "continuous_prediction"
            ]
            - ensemble_test.loc[
                indices,
                "prediction"
            ]
        )

        selected_indices = (
            fractional_parts
            .sort_values(ascending=False)
            .index[:remaining]
        )

        ensemble_test.loc[
            selected_indices,
            "prediction"
        ] += 1


# ---------------------------------------------------------
# 8. Restore submission order and validate
# ---------------------------------------------------------

ensemble_test = ensemble_test.sort_values(
    "_original_order"
).reset_index(drop=True)

ensemble_check = (
    ensemble_test
    .groupby(["road_clean", "year"])
    .agg(
        predicted_total=("prediction", "sum"),
        required_total=("annual_total", "first"),
        months=("month", "count")
    )
    .reset_index()
)

ensemble_check["difference"] = (
    ensemble_check["predicted_total"]
    - ensemble_check["required_total"]
)

print("Predictions:", len(ensemble_test))

print(
    "Incorrect annual totals:",
    (ensemble_check["difference"] != 0).sum()
)

print(
    "Road-years without 12 months:",
    (ensemble_check["months"] != 12).sum()
)

print(
    "Minimum prediction:",
    ensemble_test["prediction"].min()
)

print(
    "Maximum prediction:",
    ensemble_test["prediction"].max()
)

assert len(ensemble_test) == 15048
assert (ensemble_check["difference"] == 0).all()
assert (ensemble_check["months"] == 12).all()
assert (ensemble_test["prediction"] >= 0).all()


# ---------------------------------------------------------
# 9. Save ensemble submission
# ---------------------------------------------------------

submission_ensemble_bc = ensemble_test[
    ["prediction"]
].astype(int)

submission_ensemble_bc.to_csv(
    "/kaggle/working/submission_ensemble_B75_C25.csv",
    index=False,
    header=False
)

display(submission_ensemble_bc.head(20))

In [ ]:
import os
import base64
from IPython.display import HTML, display

submission_files = [
    "/kaggle/working/submission_catboost_annual_calibrated.csv",
    "/kaggle/working/submission_catboost_tipologia.csv",
    "/kaggle/working/submission_ensemble_B75_C25.csv"
]

buttons = []

for file_path in submission_files:
    file_name = os.path.basename(file_path)

    if not os.path.exists(file_path):
        print(f"Missing file: {file_name}")
        continue

    with open(file_path, "rb") as file:
        encoded_file = base64.b64encode(file.read()).decode()

    buttons.append(
        f"""
        <a download="{file_name}"
           href="data:text/csv;base64,{encoded_file}"
           style="
               display:block;
               width:420px;
               margin:10px 0;
               padding:12px;
               background:#1a73e8;
               color:white;
               text-decoration:none;
               border-radius:6px;
               font-weight:bold;
               text-align:center;
           ">
           Download {file_name}
        </a>
        """
    )

display(HTML("".join(buttons)))